In [ ]:
#Install packages
!pip install -q "mcp>=1.0.0,<2.0.0" mcp-types langgraph langchain-google-genai nest_asyncio

In [ ]:
#Check MCP version
import importlib.metadata

print("MCP version:", importlib.metadata.version("mcp"))

MCP version: 1.30.0


In [ ]:
!pip install -q --force-reinstall "google-auth==2.49.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.3/181.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.8/221.8 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.4/84.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 2.22.0 requires google-auth[requests]<3.0.0,>=2.56.0, but you have google-auth 2.49.0 which is incompatible.


In [ ]:
import google.auth

print("Google Auth version:", google.auth.__version__)

Google Auth version: 2.49.0


In [ ]:
#Import libraries
import os
import csv
import asyncio
import nest_asyncio

from google.colab import userdata

from langchain_core.tools import StructuredTool
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

nest_asyncio.apply()

print("All libraries imported successfully!")

All libraries imported successfully!


In [ ]:
#Load Gemini API key
os.environ["GOOGLE_API_KEY"] = userdata.get("GeminiAPIKey3")

print("Gemini API key loaded successfully!")

Gemini API key loaded successfully!


In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

In [ ]:
#Initialize Gemini
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0
)

print("Gemini initialized successfully!")

Gemini initialized successfully!


In [ ]:
#Create CSV file setup
CSV_FILE = "expenses.csv"

def _initialize_csv():
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, mode="w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(["Item", "Amount", "Category"])

print("CSV file setup ready!")

CSV file setup ready!


In [ ]:
#Create add_expense tool
def add_expense(item: str, amount: float, category: str) -> str:
    """Logs a new expense with item, amount, and category into CSV."""

    _initialize_csv()

    with open(CSV_FILE, mode="a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([item, amount, category])

    return f"Successfully logged expense: {item} - ${amount} ({category})"

print("add_expense tool created!")

add_expense tool created!


In [ ]:
#Create get_expenses tool
def get_expenses() -> str:
    """Retrieves all logged expenses from the expense tracker."""

    _initialize_csv()

    with open(CSV_FILE, mode="r") as f:
        reader = csv.reader(f)
        rows = list(reader)

    if len(rows) <= 1:
        return "No expenses recorded yet."

    return "\n".join([", ".join(row) for row in rows])

print("get_expenses tool created!")

get_expenses tool created!


In [ ]:
#Convert functions into tools
mcp_tools = [
    StructuredTool.from_function(
        func=add_expense,
        name="add_expense",
        description="Logs a new expense with item, amount, and category."
    ),

    StructuredTool.from_function(
        func=get_expenses,
        name="get_expenses",
        description="Retrieves all logged expenses from the expense tracker."
    )
]

print("Tools created successfully!")

Tools created successfully!


In [ ]:
#Create LangGraph Agent
agent = create_react_agent(llm, mcp_tools)

print("LangGraph agent created successfully!")

LangGraph agent created successfully!


/tmp/ipykernel_4437/1028513118.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, mcp_tools)


In [ ]:
#Test adding an expense
async def test_add_expense():

    prompt = "I bought a pizza for $12.50. Category is Food."

    response = await agent.ainvoke(
        {"messages": [("user", prompt)]}
    )

    for msg in response["messages"]:
        if msg.type == "ai" and msg.content:
            print("Agent:", msg.content)

await test_add_expense()

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Agent: [{'type': 'text', 'text': "I've logged your expense: **pizza** for **"}, {'type': 'text', 'text': '$12.50** under the **Food** category.', 'extras': {'signature': 'ErABCq0BARFNMg/99juFfD/NvpT80wkDZmtc41f+fNWfER3rq3ll6rp18DqdoHTfTPiFGCWhKY1LmYgEi9s7cuoV6aqUXwUyv35RM8XyjaeKJQbbSdxfVZXaUhfcyViCSHwEcHbyk0FJHSUnGZxrIvjR6ZQURxPpAbdSZB71VgFtNvw/znF/abTLDGPHquW01bqyIFoIjC/VIp31xcT9z5cfm+u0au/6OA5imkSdOPObFmg='}}]


In [ ]:
#Display all expenses
print(get_expenses())

Item, Amount, Category
pizza, 12.5, Food


In [ ]:
#Retrieve expenses using the AI agent
async def run_agentic_workflow():

    print("--- Task 1: Log an expense ---")

    prompt_1 = "I bought a pizza for $12.50. Category is Food."

    response_1 = await agent.ainvoke(
        {"messages": [("user", prompt_1)]}
    )

    for msg in response_1["messages"]:
        if msg.type == "ai" and msg.content:
            print("\n[Agent Response]:", msg.content)

    print("\n--- Task 2: Retrieve records ---")

    prompt_2 = "Show me all expenses logged so far."

    response_2 = await agent.ainvoke(
        {"messages": [("user", prompt_2)]}
    )

    for msg in response_2["messages"]:
        if msg.type == "ai" and msg.content:
            print("\n[Agent Response]:", msg.content)


await run_agentic_workflow()

--- Task 1: Log an expense ---


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[Agent Response]: [{'type': 'text', 'text': 'I have logged your expense of **$12.50** for **pizza** under the **Food** category.', 'extras': {'signature': 'EsoBCscBARFNMg+OWEacR4RvD1N2DNqpnl0qVAFg7Q3XMwQzKxTJH7CPR02fGsCmW98uKGVT3utNU/scYVUMjJwew8eOtHwtTooKe9C38imgQ1V7Y7x1r8CLPyOO+EJCDXMxwrz/Qvb32JSuEe08gvEAXEkY2QDCLck9AiRainIJvXzckdkN83ojvjoojQcqrp78UQufQZpxy7oQIs1vXrTLInDZDL6y0WW/1vOntrVOQRlbaBwsqVUGZDwPhSJOrRvHpwJQ8Af4qHWmGA=='}}]

--- Task 2: Retrieve records ---


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[Agent Response]: [{'type': 'text', 'text': 'Here are all the expenses logged so far:\n\n* **Item:** Pizza | **Amount:** $12.50 | **Category:** Food\n* **Item:** Pizza | **Amount:** $12.50 | **Category:** Food\n* **Item:** Pizza | **Amount:** $12.50 | **Category:** Food\n\n**Total:** $37.50', 'extras': {'signature': 'EsECCr4CARFNMg8JpHZTYjA0Z5zGW3CGDkXMrHknCpp6LL+2J3fOcyyXhHUMtpBWhoP6zmCN1h0zv4WxDAZ59LEkdFI7cCrLijoYLR0kmZuajkP59qkWFh4FCdYsRQmQfVp9zHulpwxF0MJapbaAIMaoS/32Vpwy/cNDYuZa2T2d5iKuBKxGwrKsu/GPoQuS3zq+uIk9MOxZDNQZnriTdYCfxLUN0lBtgfe30o9DHZ/zCFvy0FHzp6kUX99VwS23oJ8qqY3EKE08jfch4IMgHOPco5VixzHfE5MSpgXPizM1RsAREljkY1bWr3FBoiSIOg99g8TuL3bfaB9WGvPut/Ln7oR9BVLselWVre27Ki6DGXhglkV7BzANESxrei40kJDR1m1a0kEAQYBYxe0N/go2aLvvgOqFr6htp96C1g6YsqxG'}}]
